<a href="https://colab.research.google.com/github/Gauravd1710/legal-doc-analyzer/blob/main/notebooks/02_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

BASE      = '/content/drive/MyDrive/LegalDocAnalyzer'
REPO_PATH = '/content/legal-doc-analyzer'

sys.path.append(f'{REPO_PATH}/src')

print("✅ Drive mounted")
print(f"BASE      : {BASE}")
print(f"REPO_PATH : {REPO_PATH}")

Mounted at /content/drive
✅ Drive mounted
BASE      : /content/drive/MyDrive/LegalDocAnalyzer
REPO_PATH : /content/legal-doc-analyzer


In [3]:
# datasets    → download CUAD & LEDGAR from HuggingFace Hub
# pandas      → inspect and filter data in table form
# sklearn     → train/val/test splitting

!pip install datasets pandas scikit-learn -q

from datasets import load_dataset
import pandas as pd
import json, os
from sklearn.model_selection import train_test_split

print("✅ Libraries ready")

✅ Libraries ready


In [4]:
import json

with open(f'{BASE}/data/raw/CUAD_v1.json') as f:
    raw = json.load(f)

# CUAD is in SQuAD format — data → paragraphs → qas
rows = []
for article in raw['data']:
    title = article['title']
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            rows.append({
                'title'   : title,
                'question': qa['id'].split('__')[-1].replace('_', ' '),
                'context' : context,
                'answers' : {
                    'text'        : [a['text'] for a in qa['answers']],
                    'answer_start': [a['answer_start'] for a in qa['answers']]
                }
            })

import pandas as pd
df_cuad = pd.DataFrame(rows)

print(f"✅ Loaded CUAD manually")
print(f"Total rows : {len(df_cuad)}")
print(f"\nColumns: {df_cuad.columns.tolist()}")
print(f"\nSample:")
print(df_cuad.iloc[0])

✅ Loaded CUAD manually
Total rows : 20910

Columns: ['title', 'question', 'context', 'answers']

Sample:
title       LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...
question                                        Document Name
context     EXHIBIT 10.6\n\n                              ...
answers     {'text': ['DISTRIBUTOR AGREEMENT'], 'answer_st...
Name: 0, dtype: object


In [5]:
# verify the structure looks right

print("=== SHAPE ===")
print(f"Rows: {len(df_cuad)}, Columns: {df_cuad.columns.tolist()}")

print("\n=== SAMPLE ANSWER STRUCTURE ===")
sample = df_cuad.iloc[0]
print(f"Title    : {sample['title'][:60]}")
print(f"Question : {sample['question']}")
print(f"Context  : {sample['context'][:200]}")
print(f"Answer   : {sample['answers']}")

print("\n=== NULL CHECK ===")
print(df_cuad.isnull().sum())

print("\n=== ANSWER COUNT ===")
df_cuad['has_answer'] = df_cuad['answers'].apply(
    lambda x: len(x['text']) > 0
)
print(f"Rows WITH answers    : {df_cuad['has_answer'].sum()}")
print(f"Rows WITHOUT answers : {(~df_cuad['has_answer']).sum()}")
print(f"Usable rows          : {df_cuad['has_answer'].sum()}")

=== SHAPE ===
Rows: 20910, Columns: ['title', 'question', 'context', 'answers']

=== SAMPLE ANSWER STRUCTURE ===
Title    : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Question : Document Name
Context  : EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Com
Answer   : {'text': ['DISTRIBUTOR AGREEMENT'], 'answer_start': [44]}

=== NULL CHECK ===
title       0
question    0
context     0
answers     0
dtype: int64

=== ANSWER COUNT ===
Rows WITH answers    : 6702
Rows WITHOUT answers : 14208
Usable rows          : 6702


In [6]:
# WHY: Need to see exact question strings in THIS version
# of CUAD so our keyword mapping in Cell 6 matches correctly

questions = df_cuad['question'].unique()
print(f"Total unique question types: {len(questions)}\n")
for i, q in enumerate(questions):
    print(f"{i+1:02d}. {q}")

Total unique question types: 41

01. Document Name
02. Parties
03. Agreement Date
04. Effective Date
05. Expiration Date
06. Renewal Term
07. Notice Period To Terminate Renewal
08. Governing Law
09. Most Favored Nation
10. Non-Compete
11. Exclusivity
12. No-Solicit Of Customers
13. Competitive Restriction Exception
14. No-Solicit Of Employees
15. Non-Disparagement
16. Termination For Convenience
17. Rofr/Rofo/Rofn
18. Change Of Control
19. Anti-Assignment
20. Revenue/Profit Sharing
21. Price Restrictions
22. Minimum Commitment
23. Volume Restriction
24. Ip Ownership Assignment
25. Joint Ip Ownership
26. License Grant
27. Non-Transferable License
28. Affiliate License-Licensor
29. Affiliate License-Licensee
30. Unlimited/All-You-Can-Eat-License
31. Irrevocable Or Perpetual License
32. Source Code Escrow
33. Post-Termination Services
34. Audit Rights
35. Uncapped Liability
36. Cap On Liability
37. Liquidated Damages
38. Warranty Duration
39. Insurance
40. Covenant Not To Sue
41. Third Pa

In [7]:
ENTITY_MAP = {
    "PARTY": [
        "parties", "licensor", "licensee",
        "counterparty", "document name",
        "license grant"                      # ← rescued
    ],
    "DATE": [
        "effective date", "expiration date",
        "renewal term", "notice period",
        "agreement date"
    ],
    "AMOUNT": [
        "revenue", "payment", "price", "fee",
        "cap on liability", "minimum commitment",
        "volume restriction", "post-termination",
        "uncapped liability",                 # ← rescued
        "liquidated damages",                 # ← rescued
        "most favored nation"                 # ← rescued
    ],
    "JURISDICTION": [
        "governing law", "jurisdiction"
    ],
    "TERM": [
        "termination", "indemnification", "confidentiality",
        "non-compete", "ip ownership", "warranty",
        "non-disparagement", "audit rights",
        "exclusivity",                        # ← rescued
        "covenant not to sue",                # ← rescued
        "no-solicit of employees",            # ← rescued
        "no-solicit of customers",            # ← rescued
        "competitive restriction"             # ← rescued
    ]
}

df_useful = df_cuad[df_cuad['has_answer']].copy()

print("=== UPDATED ENTITY MAPPING ===\n")
total_matched = 0

for entity, keywords in ENTITY_MAP.items():
    mask = df_useful['question'].str.lower().apply(
        lambda q: any(kw in q.lower() for kw in keywords)
    )
    count = mask.sum()
    total_matched += count
    print(f"{entity:15s} → {count:5d} rows")

print(f"\nTotal mapped rows : {total_matched}")
print(f"Unmapped rows     : {len(df_useful) - total_matched}")

=== UPDATED ENTITY MAPPING ===

PARTY           →  1356 rows
DATE            →  1560 rows
AMOUNT          →  1085 rows
JURISDICTION    →   437 rows
TERM            →  1430 rows

Total mapped rows : 5868
Unmapped rows     : 834


In [8]:
def assign_entity_label(question):
    q = question.lower()
    for entity, keywords in ENTITY_MAP.items():
        if any(kw in q for kw in keywords):
            return entity
    return None

df_useful['entity_label'] = df_useful['question'].apply(assign_entity_label)

before = len(df_useful)
df_useful = df_useful[df_useful['entity_label'].notna()].copy()
after = len(df_useful)

print("=== UPDATED LABEL DISTRIBUTION ===\n")
counts = df_useful['entity_label'].value_counts()
print(counts)

total = counts.sum()
print("\n% distribution:")
for label, count in counts.items():
    bar = "█" * int(count / total * 40)
    print(f"{label:15s} {count:5d}  {bar}")

min_count = counts.min()
max_count = counts.max()
ratio = max_count / min_count

print(f"\nRows kept      : {after}")
print(f"Rows dropped   : {before - after}")
print(f"Imbalance ratio: {ratio:.1f}x  ", end="")
if ratio > 3:
    print("⚠️  Still imbalanced — will handle in Step 3")
else:
    print("✅ Balanced enough")

=== UPDATED LABEL DISTRIBUTION ===

entity_label
DATE            1560
PARTY           1356
TERM            1248
AMOUNT          1085
JURISDICTION     437
Name: count, dtype: int64

% distribution:
DATE             1560  ██████████
PARTY            1356  █████████
TERM             1248  ████████
AMOUNT           1085  ███████
JURISDICTION      437  ███

Rows kept      : 5686
Rows dropped   : 1016
Imbalance ratio: 3.6x  ⚠️  Still imbalanced — will handle in Step 3


In [9]:
# WHY: Sanity check — make sure each entity label
# has sensible context text and answer text

print("=== ONE SAMPLE PER ENTITY TYPE ===\n")

for entity in ENTITY_MAP.keys():
    sample = df_useful[df_useful['entity_label'] == entity].iloc[0]
    print(f"{'='*50}")
    print(f"ENTITY    : {entity}")
    print(f"QUESTION  : {sample['question']}")
    print(f"ANSWER    : {sample['answers']['text'][0][:100]}")
    print(f"CONTEXT   : {sample['context'][:150]}")
    print()

=== ONE SAMPLE PER ENTITY TYPE ===

ENTITY    : PARTY
QUESTION  : Document Name
ANSWER    : DISTRIBUTOR AGREEMENT
CONTEXT   : EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between El

ENTITY    : DATE
QUESTION  : Agreement Date
ANSWER    : 7th day of September, 1999.
CONTEXT   : EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between El

ENTITY    : AMOUNT
QUESTION  : Price Restrictions
ANSWER    : The Company  also  reserves  the right to                            increase  or  decrease  the pri
CONTEXT   : EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between El

ENTITY    : JURISDICTION
QUESTION  : Governing Law
ANSWER    : This Agreement is to be construed according to the laws          of the State of Ill